# Surface defect demo: shared inference code
Sarthak Jain · 23f3000839 · T3 2026

This companion publishes the inference code used by my course demonstration, for all competition participants under the MIT license. It contains the three architecture definitions, grayscale normalization, probability averaging, component filtering and column-major RLE. It contains no competition images or labels.

The registered private submission notebook is maintained separately for instructor access. The demo and its selected weights are available at https://huggingface.co/spaces/sarthak-wiz01/surface-defects-t32026-23f3000839. This code-sharing notebook does not make a leaderboard submission.

Competition associated with this code: [SurfaceGuard AI: Industrial Surface Anomaly Segmentation](https://www.kaggle.com/competitions/industrial-surface-defect-detection). The notebook shares my own model/inference definitions. Competition inputs are not attached.

In [ ]:
from pathlib import Path
import sys


In [ ]:
%%writefile /kaggle/working/final_pipeline.py
"""Inference definitions extracted from the trained model source."""

import numpy as np
import torch
from torch import nn
from scipy import ndimage

CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
H, W = 128, 800


def encode_rle(mask):
    flat = mask.astype(np.uint8).ravel(order="F")
    changes = np.flatnonzero(np.diff(np.r_[0, flat, 0])) + 1
    changes[1::2] -= changes[::2]
    return " ".join(map(str, changes)) or " "


def remove_components(mask, min_area):
    if not min_area:
        return mask
    labels, n = ndimage.label(mask)
    sizes = np.bincount(labels.ravel())
    keep = sizes >= min_area
    keep[0] = False
    return keep[labels]


class Block(nn.Module):
    def __init__(self, a, b):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(a, b, 3, padding=1, bias=False),
            nn.BatchNorm2d(b),
            nn.ReLU(),
            nn.Conv2d(b, b, 3, padding=1, bias=False),
            nn.BatchNorm2d(b),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class ScratchUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.ModuleList(
            [Block(1, 24), Block(24, 48), Block(48, 96), Block(96, 192)]
        )
        self.dec = nn.ModuleList([Block(288, 96), Block(144, 48), Block(72, 24)])
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Conv2d(24, 4, 1)

    def forward(self, x):
        skips = []
        for i, b in enumerate(self.enc):
            x = b(x)
            skips.append(x)
            if i < 3:
                x = self.pool(x)
        for skip, b in zip(skips[-2::-1], self.dec):
            x = b(
                torch.cat(
                    [
                        nn.functional.interpolate(
                            x,
                            size=skip.shape[-2:],
                            mode="bilinear",
                            align_corners=False,
                        ),
                        skip,
                    ],
                    dim=1,
                )
            )
        return self.head(x)


def make_model(name, pretrained=True):
    if name == "scratch_unet":
        return ScratchUNet()
    import segmentation_models_pytorch as smp

    if name == "resnet34_unet":
        return smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet" if pretrained else None,
            in_channels=1,
            classes=4,
        )
    if name == "efficientnet_b0_fpn":
        return smp.FPN(
            encoder_name="efficientnet-b0",
            encoder_weights="imagenet" if pretrained else None,
            in_channels=1,
            classes=4,
            decoder_pyramid_channels=128,
            decoder_segmentation_channels=64,
        )
    raise ValueError(name)


In [ ]:
%%writefile /kaggle/working/predict_final.py
"""Load a trained final-pipeline checkpoint and predict consistent masks.

This module is shared by the Kaggle inference notebook and optional Gradio app.
"""

from pathlib import Path
import json
import numpy as np
from PIL import Image
import torch
from final_pipeline import CLASSES, H, W, make_model, remove_components, encode_rle


class Segmenter:
    def __init__(self, checkpoint, selection, device=None):
        self.device = torch.device(
            device
            or (
                "cuda"
                if torch.cuda.is_available()
                else "mps" if torch.backends.mps.is_available() else "cpu"
            )
        )
        self.selection = json.loads(Path(selection).read_text())
        checkpoint = Path(checkpoint)
        names = self.selection.get("model_names", [self.selection["model"]])
        self.models = []
        for name in names:
            if len(names) > 1 and not checkpoint.is_dir():
                raise ValueError("An ensemble requires a directory of checkpoints")
            if checkpoint.is_dir():
                nested = checkpoint / name / "best.pt"
                path = nested if nested.is_file() else checkpoint / f"{name}.pt"
            else:
                path = checkpoint
            saved = torch.load(path, map_location=self.device, weights_only=False)
            if saved["architecture"] != name:
                raise ValueError("Checkpoint and selection architecture disagree")
            model = make_model(name, pretrained=False).to(self.device)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            self.models.append(model)

    @torch.inference_mode()
    def predict_batch(self, images, tta=None):
        if tta is None:
            tta = self.selection.get("tta", False)
        arrays = []
        for image in images:
            if isinstance(image, (str, Path)):
                with Image.open(image) as f:
                    a = np.asarray(f.convert("L"), dtype=np.float32) / 255
            else:
                raw = np.asarray(image)
                integer_input = np.issubdtype(raw.dtype, np.integer)
                a = raw.astype(np.float32)
                if a.ndim == 3:
                    rgb = a if integer_input or a.max() > 1 else a * 255
                    a = (
                        np.asarray(
                            Image.fromarray(
                                np.rint(rgb).clip(0, 255).astype("uint8")
                            ).convert("L"),
                            dtype=np.float32,
                        )
                        / 255
                    )
                elif integer_input or a.max() > 1:
                    a = a / 255
            if a.shape != (H, W):
                raise ValueError(
                    f"Expected an original {H}x{W} strip; got {a.shape}. Resizing would distort the measurement geometry."
                )
            arrays.append((a - 0.5) / 0.25)
        x = torch.from_numpy(np.stack(arrays)[:, None]).to(self.device)
        probabilities = []
        for model in self.models:
            p = model(x).sigmoid()
            if tta:
                p = (p + torch.flip(model(torch.flip(x, [-1])).sigmoid(), [-1])) / 2
            # Validation stores each model's probabilities as float16 before averaging.
            probabilities.append(p.float().cpu().numpy().astype(np.float16))
        p = (
            sum(value.astype(np.float32) for value in probabilities)
            / len(probabilities)
        ).astype(np.float16)
        masks = np.empty_like(p, dtype=bool)
        for i in range(len(p)):
            for c in range(4):
                masks[i, c] = remove_components(
                    p[i, c] >= self.selection["thresholds"][c],
                    self.selection["min_areas"][c],
                )
        return masks, p

    def rles(self, images):
        masks, _ = self.predict_batch(images)
        return [
            {c: encode_rle(mask) for c, mask in zip(CLASSES, item)} for item in masks
        ]


In [ ]:
sys.path.insert(0, "/kaggle/working")
from predict_final import Segmenter
from final_pipeline import CLASSES, H, W

print("Mask classes:", CLASSES, "native size:", H, W)


## Use the browser-exported ensemble
The optional recipe below requires internet, onnxruntime, NumPy and Pillow. The published bundle stores ONNX models and the frozen selection under `models/`.

```python
import json, numpy as np, onnxruntime as ort
from PIL import Image
from huggingface_hub import snapshot_download
from final_pipeline import remove_components
assets = Path(snapshot_download(repo_id="sarthak-wiz01/surface-defects-t32026-23f3000839", repo_type="space")) / "models"
config = json.loads((assets / "final_selection.json").read_text())
image = np.asarray(Image.open("original_800x128_strip.png").convert("L"), dtype=np.float32)
x = ((image / 255 - 0.5) / 0.25)[None, None]
assert x.shape == (1, 1, 128, 800)
probabilities = []
for name in config["model_names"]:
    session = ort.InferenceSession(str(assets / (name + ".onnx")))
    logits = session.run(None, {"image": x})[0]
    probabilities.append((1 / (1 + np.exp(-logits))).astype(np.float16))
p = (sum(a.astype(np.float32) for a in probabilities) / len(probabilities)).astype(np.float16)[0]
masks = np.stack([remove_components(p[c] >= config["thresholds"][c], config["min_areas"][c]) for c in range(4)])
```

The code below is the browser implementation deployed with the same models. No competition inputs are attached to this notebook.

## Browser inference implementation
```javascript
'use strict';
const W=800,H=128,N=W*H,classes=['defect_a','defect_b','defect_c','defect_d'];
const colors=[[235,70,60],[55,190,100],[60,130,240],[220,170,40]];
const el=id=>document.getElementById(id),status=el('status'),run=el('run');
const original=el('original'),overlay=el('overlay');
let pixels=null,masks=null,decision=null;
const sessions=new Map();
const visible=classes.map(()=>true);
ort.env.wasm.wasmPaths=new URL('vendor/',location.href).href;
ort.env.wasm.numThreads=1;
function message(text,error=false){status.textContent=text;status.className=error?'error':'';}
function halfRound(value){
 if(value===0)return 0;
 const step=2**Math.max(-24,Math.floor(Math.log2(value))-10),scaled=value/step;
 const lower=Math.floor(scaled),fraction=scaled-lower;
 return (lower+(fraction>.5||(fraction===.5&&lower%2)?1:0))*step;
}
function removeSmall(mask,area){
 if(!area)return mask;
 const visited=new Uint8Array(N),queue=new Int32Array(N);
 for(let start=0;start<N;start++){
  if(!mask[start]||visited[start])continue;
  let head=0,end=1;queue[0]=start;visited[start]=1;
  while(head<end){
   const i=queue[head++],x=i%W;
   const neighbors=[i-W,i+W,x>0?i-1:-1,x<W-1?i+1:-1];
   for(const j of neighbors)if(j>=0&&j<N&&mask[j]&&!visited[j]){visited[j]=1;queue[end++]=j;}
  }
  if(end<area)for(let i=0;i<end;i++)mask[queue[i]]=0;
 }
 return mask;
}
function render(){
 if(!pixels)return;
 const output=new ImageData(new Uint8ClampedArray(pixels.data),W,H);
 if(masks)for(let c=0;c<4;c++)if(visible[c])for(let i=0;i<N;i++)if(masks[c][i]){
  for(let k=0;k<3;k++)output.data[4*i+k]=Math.trunc(.55*output.data[4*i+k]+.45*colors[c][k]);
 }
 overlay.getContext('2d').putImageData(output,0,0);
}
classes.forEach((name,c)=>{
 const label=document.createElement('label'),check=document.createElement('input'),dot=document.createElement('span');
 check.type='checkbox';check.checked=true;check.addEventListener('change',()=>{visible[c]=check.checked;render();});
 dot.className='dot';dot.style.background=`rgb(${colors[c].join(',')})`;
 label.append(check,dot,document.createTextNode(name));el('legend').append(label);
});
el('file').addEventListener('change',async()=>{
 run.disabled=true;masks=null;pixels=null;el('results').hidden=true;el('empty').hidden=false;
 const file=el('file').files[0];if(!file){message('Choose an image to begin.');return;}
 const url=URL.createObjectURL(file),image=new Image();
 try{
  await new Promise((resolve,reject)=>{image.onload=resolve;image.onerror=reject;image.src=url;});
  if(image.naturalWidth!==W||image.naturalHeight!==H)throw new Error(`Use an original 800 × 128 image. Received ${image.naturalWidth} × ${image.naturalHeight}.`);
  const context=original.getContext('2d',{willReadFrequently:true});context.drawImage(image,0,0);pixels=context.getImageData(0,0,W,H);
  render();run.disabled=false;message(`${file.name} · ready to inspect.`);
 }catch(error){message(error.message||'The image could not be opened.',true);}
 finally{URL.revokeObjectURL(url);}
});
async function modelSession(name,index,total){
 if(sessions.has(name))return sessions.get(name);
 message(`Loading model ${index+1} of ${total}: ${name.replaceAll('_',' ')}…`);
 const session=await ort.InferenceSession.create(`models/${name}.onnx`,{executionProviders:['wasm'],graphOptimizationLevel:'all'});
 sessions.set(name,session);return session;
}
run.addEventListener('click',async()=>{
 if(!pixels)return;run.disabled=true;
 try{
  decision??=await (await fetch('models/final_selection.json')).json();
  const input=new Float32Array(N);
  for(let i=0;i<N;i++){
   const gray=Math.round(.299*pixels.data[4*i]+.587*pixels.data[4*i+1]+.114*pixels.data[4*i+2]);
   input[i]=(gray/255-.5)/.25;
  }
  const tensor=new ort.Tensor('float32',input,[1,1,H,W]),sum=new Float32Array(4*N);
  const start=performance.now();
  for(let c=0;c<decision.model_names.length;c++){
   const name=decision.model_names[c],session=await modelSession(name,c,decision.model_names.length);
   message(`Running model ${c+1} of ${decision.model_names.length}…`);
   await new Promise(resolve=>requestAnimationFrame(resolve));
   const outputs=await session.run({image:tensor}),values=outputs.logits.data;
   for(let i=0;i<values.length;i++)sum[i]+=halfRound(1/(1+Math.exp(-values[i])));
  }
  masks=classes.map((_,c)=>{
   const mask=new Uint8Array(N),threshold=halfRound(decision.thresholds[c]);
   for(let i=0;i<N;i++)mask[i]=halfRound(sum[c*N+i]/decision.model_names.length)>=threshold?1:0;
   return removeSmall(mask,decision.min_areas[c]);
  });
  render();const body=el('results').querySelector('tbody');body.replaceChildren();
  classes.forEach((name,c)=>{const count=masks[c].reduce((a,b)=>a+b,0),row=body.insertRow();[name,count.toLocaleString(),(100*count/N).toFixed(3)+'%'].forEach(value=>{row.insertCell().textContent=value;});});
  el('empty').hidden=true;el('results').hidden=false;
  window.lastPrediction={pixel_counts:masks.map(mask=>mask.reduce((a,b)=>a+b,0)),seconds:(performance.now()-start)/1000};
  message(`Complete · ${window.lastPrediction.seconds.toFixed(1)} seconds. Toggle a category to inspect its overlay.`);
 }catch(error){message('Inference failed: '+error.message,true);console.error(error);}
 finally{run.disabled=false;}
});
window.surfaceHelpers={halfRound,removeSmall};

```